# Bloque 2 — Simulador interactivo de Merton Jump-Diffusion

Objetivo: ajustar los parámetros del proceso y observar cómo cambia una trayectoria de precios.

Proceso de Merton (1976):

\[
\frac{dS_t}{S_t} = (\mu - \lambda k)dt + \sigma dW_t + (J - 1)dN_t
\]

Donde:

- λ: frecuencia de saltos por año.
- J: multiplicador del salto (log-normal).
- k = E[J − 1]: compensación del drift.
- σ: volatilidad de difusión.

**Guion breve:** pida predicciones antes de mover cada control. Después ajuste λ, μ_J, σ_J y σ y compare GBM puro con Merton.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def simular_merton(S0=100, mu=0.05, sigma=0.2, lam=1.0, mu_J=-0.08, sigma_J=0.12, T=1, steps=252, seed=42):
    np.random.seed(seed)
    dt = T / steps
    prices = np.zeros(steps + 1)
    prices[0] = S0
    saltos = 0
    k = np.exp(mu_J + 0.5 * sigma_J**2) - 1
    for t in range(1, steps + 1):
        Z = np.random.normal(0, 1)
        N = np.random.poisson(lam * dt)
        if N > 0:
            saltos += N
        J = np.random.lognormal(mu_J, sigma_J) if N > 0 else 1
        prices[t] = prices[t-1] * np.exp((mu - lam * k - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z) * (J ** N)

    plt.figure(figsize=(14, 5))
    plt.plot(prices, lw=1.8, color='black', label='Merton Jump-Diffusion')
    plt.axhline(y=S0, color='gray', linestyle='--', alpha=0.4)
    plt.title(f"λ={lam:.2f} | μ_J={mu_J:.2f} | σ_J={sigma_J:.2f} | σ={sigma:.2f} | Saltos: {saltos}", fontsize=14)
    plt.xlabel("Días")
    plt.ylabel("Precio")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

interact(
    simular_merton,
    lam=FloatSlider(min=0, max=5, step=0.1, value=1.0, description='λ (saltos/año)'),
    mu_J=FloatSlider(min=-0.3, max=0.3, step=0.01, value=-0.08, description='μ_J'),
    sigma_J=FloatSlider(min=0, max=0.3, step=0.01, value=0.12, description='σ_J'),
    sigma=FloatSlider(min=0.1, max=0.5, step=0.01, value=0.2, description='σ (difusión)')
);

In [ ]:
# Comparación automática: GBM puro (λ=0) vs Merton (λ>0)
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, lam, title in zip(axes, [0, 1.0], ['GBM puro (λ=0)', 'Merton (λ=1.0)']):
    np.random.seed(7)
    dt = 1 / 252
    prices = np.zeros(253)
    prices[0] = 100
    for t in range(1, 253):
        Z = np.random.normal()
        N = np.random.poisson(lam * dt)
        J = np.random.lognormal(-0.08, 0.12) if N > 0 else 1
        prices[t] = prices[t-1] * np.exp((0.05 - 0.5 * 0.2**2) * dt + 0.2 * np.sqrt(dt) * Z) * J**N
    ax.plot(prices)
    ax.set_title(title)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Preguntas de análisis

1. ¿qué cambio esperaba al aumentar λ?
2. ¿la trayectoria confirmó mi predicción?
3. ¿qué ajusté para comprobar otra hipótesis?
4. ¿qué resultado me hizo revisar mi razonamiento?

**Preguntas de facilitación**

- ¿Qué parámetro controla la probabilidad de al menos un salto?
- ¿Qué significa μ_J negativo?
- ¿Por qué una trayectoria sin saltos no demuestra que λ=0?